# S6E9 | LR-Margin GBDT + OOF Stack | LB 0.94675

*Generator-aware ridge logistic regression (heuljax) → GBDT boosted from its logit → honest out-of-fold rank stack with public models.*

This notebook is the pipeline behind my best submission (public **0.94675**, OOF **0.94664** on `StratifiedKFold(5, shuffle=True, random_state=42)`).
It is built on other people's work, so the first thing you should know is **what is borrowed and what is mine**:

| Part | Component | Origin | What I changed or added |
|---|---|---|---|
| A | Generator-aware feature basis (GPT-2 BPE token hierarchy of income, beta-chain target rates, token × context cells, category pairs, transductive composition features) + L2 ridge logistic regression (LBFGS) | **[heuljax — KPS6E09 Generator-Aware Ridge Logistic Regression](https://www.kaggle.com/code/heuljax/kps6e09-generator-aware-ridge-logistic-regression)**, ported almost verbatim | 5 outer folds instead of 10; **inner-CV logits for the training rows** so the GLM can serve as a base margin; a decimal floor chain (`income//1000 → //100 → exact`) and per-value original-data means as extra features; `L2 = 30`. Each of these is worth ≤ +0.00001 — the basis is heuljax's. |
| B | **LR-margin residual GBDT**: LightGBM (`init_score`) and XGBoost (`base_margin`) boosted *from the GLM's logit* on my 25 target-encoded keys | mine | New. On the same folds: GLM alone 0.94632 → residual LightGBM **0.94646** / XGBoost 0.94645; my previous plain GBDT on these features was 0.94614. |
| C | Public out-of-fold + test predictions used as stack members | [heuljax — XGB Sample](https://www.kaggle.com/code/heuljax/kps6e09-xgb-sample), [heuljax — ridge LR](https://www.kaggle.com/code/heuljax/kps6e09-generator-aware-ridge-logistic-regression) (10-fold outputs), [BlamerX — XGBoost + Window Encodings](https://www.kaggle.com/code/blamerx/s6e9-xgboost-window-encodings-0-946-cv), [megayak — six feature views OOF library](https://www.kaggle.com/datasets/megayak/s6e9-six-feature-views-oof-library) (views A, D and the 3-seed RealMLP retrain of [yekenot's RealMLP](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch)) | Used as published, attached as notebook/dataset inputs. Nothing is re-trained. |
| D | **Honest stacking**: id-aligned rank blend, greedy hill-climb on OOF AUC, weights fitted inside a 5-fold nested loop and averaged, plus a fast Mann–Whitney AUC | mine | New. Nested gain over the best single member: +0.00023, positive in 5/5 folds. |
| D | Four boundary rules (income ≥ \$170,537 → top; income 31,004–41,970, commute ≥ 83 km, and the \$30,000 / no-subsidy / low-concern-or-anxious cell → bottom) | thresholds from [Chris Deotte's EDA](https://www.kaggle.com/code/cdeotte/fable-5-1-eda-original-data-insights) and community discussion; the rank-shift form from [taeyangg4](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master) / [lucifer19](https://www.kaggle.com/code/lucifer19/ev-grand-prix-48-engine-cpu-pit-stop-blend) | I verified them on OOF (+0.000002, harmless) and kept them. |

What I found along the way (details in the closing section):

* **The strongest single model on this competition is a linear model.** heuljax's ridge GLM scores 0.94640 (10-fold) with ~330 engineered rate features — above every tree model on the board — and it correlates with my GBDTs at only 0.991 (tree models correlate with each other at 0.998+). The first GPT-2 token of the income string is its single strongest coefficient; a decimal hierarchy (leading digits, `//1000`, `//100`) or the cl100k tokenizer is measurably worse, so the generator really did leave a BPE-shaped fingerprint.
* **Boosting from the GLM's logit beats blending with it.** A GBDT with the GLM logit as `init_score`/`base_margin` gains +0.00015 over the GLM and +0.0003 over the same GBDT without the margin, on the same folds, with 3× fewer trees.
* **Diversity comes from feature views, not learners.** In the honest stack, members that see the data differently (window rates, no-exact-key ladder, RealMLP) get weight even when individually weaker; an XGBoost on the same features as a LightGBM gets none.
* **The public board at 0.9465+ is mostly one file.** The 0.94657–0.94663 public notebooks are re-blends of one lineage (some of it measured against the public split itself), so I used none of them as an anchor. Everything here is weighted on out-of-fold predictions only.

| Pipeline | 5-fold OOF | Public LB |
|---|---|---|
| My previous best: 25 TE keys, LightGBM + shallow XGBoost + focal LGBM | 0.94614 | 0.94637 |
| heuljax ridge GLM, ported, 5 folds (10 folds: 0.94640) | 0.94632 | — |
| GLM-margin residual LightGBM / XGBoost (Part B) | 0.94646 / 0.94645 | — |
| Honest rank stack of the six public OOF models alone (no model of mine) | 0.94662 | 0.94670 |
| **+ the residual GBDTs (this notebook; its `submission.csv`, run end-to-end, scored 0.94675)** | **0.94664** | **0.94675** |
| + a heuljax-basis XGBoost with the GLM margin, 3-seed GLM, 3-seed MLP (my later runs) | 0.94667 | 0.94674 |

A full run of this notebook gives GLM 0.94634, residual LightGBM 0.94647 / XGBoost 0.94646, stack 0.94663, and its `submission.csv` scored **0.94675** on the public board. Differences of 0.00001–0.00003 on the public board are noise (paired standard error ≈ 0.00002 for 20 % of 287k rows); the OOF numbers are the ones to read.

**Inputs to attach** (Add Input): competition `playground-series-s6e9`; dataset `itzzomkar/ev-adoption-behavior-and-range-anxiety`; dataset `megayak/s6e9-six-feature-views-oof-library`; notebook outputs of `heuljax/kps6e09-generator-aware-ridge-logistic-regression`, `heuljax/kps6e09-xgb-sample`, `blamerx/s6e9-xgboost-window-encodings-0-946-cv`. Internet on (the GPT-2 tokenizer is downloaded). Missing public inputs are skipped with a warning and the stack uses whatever is attached. Runtime on a T4: roughly 40 minutes.

## 0. Setup

In [ ]:
import os, gc, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
import lightgbm as lgb
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
warnings.filterwarnings("ignore")

SEED, FOLDS, INNER_FOLDS = 42, 5, 5
TARGET, ID = "Will_Buy_EV", "id"
DEBUG_ROWS = int(os.getenv("DEBUG_ROWS", "0"))          # >0: subsample for a quick smoke test
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
XGB_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))
print("device:", DEVICE, "| cpus:", os.cpu_count())

def find_file(name, hint=None):
    # Kaggle mounts inputs under /kaggle/input/<type>/<owner>/<slug>/...; search by file name (+ optional path hint).
    hits = []
    for root in [Path("/kaggle/input"), Path("data"), Path("../data"), Path("ref2"), Path("../ref2")]:
        if root.is_dir():
            hits += [p for p in root.rglob(name) if p.is_file() and (hint is None or hint in str(p).lower())]
    if not hits:
        raise FileNotFoundError(f"{name} (hint={hint})")
    return str(sorted(hits)[0])

## 1. Data

In [ ]:
train = pd.read_csv(find_file("train.csv", "playground-series-s6e9") if Path("/kaggle/input").is_dir() else find_file("train.csv"))
test = pd.read_csv(find_file("test.csv", "playground-series-s6e9") if Path("/kaggle/input").is_dir() else find_file("test.csv"))
orig = pd.read_csv(find_file("EV_Adoption_and_Range_Anxiety_Dataset.csv")).drop(columns=["Buyer_ID"]).dropna()
if DEBUG_ROWS:
    train = train.sample(DEBUG_ROWS, random_state=0).reset_index(drop=True)
    test = test.sample(DEBUG_ROWS // 3, random_state=0).reset_index(drop=True)
train = train.reset_index(drop=True); test = test.reset_index(drop=True)
y = (train[TARGET] == "Yes").astype(np.int8).to_numpy()
orig["_y"] = (orig[TARGET] == "Yes").astype(float)
TRAIN_IDS, TEST_IDS = train[ID].to_numpy(), test[ID].to_numpy()
SPLITS = list(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(train)), y))
print(f"train {train.shape}  test {test.shape}  orig {orig.shape}  positive rate {y.mean():.4f}")

## Part A — Referenced: heuljax's generator-aware ridge logistic regression (ported)

Everything in this part is [heuljax's](https://www.kaggle.com/code/heuljax/kps6e09-generator-aware-ridge-logistic-regression) design; I have kept the code structure and names so you can diff it against the original. In one paragraph, what it does:

* The income value is written as a string and split into **GPT-2 BPE tokens** (` 92887` → ` 9`, `28`, `87`; ` 30000` → ` 3`, `0000`; ` 170537` → ` 17`, `05`, `37`). The first token (`L1`), the first two (`L2`), the exact value (`IV`) and the last token (`LAST`) form a hierarchy. A **beta chain** shrinks each level's buyer rate toward its parent's (`prior → L1 → L2 → IV`) instead of toward the global mean, at strengths 5 / 20 / 80.
* **Cells**: token × context (city, subsidy, concern, anxiety, …), commute-integer × context, the generator gate cell (`subsidy × concern × anxiety × home charging`) × token / commute / income quantile, category pairs, category × income/commute bins. Every cell is a smoothed buyer rate on the logit scale.
* **Label-free features**: raw numerics, hinge splines at five quantiles, one-hots, train+test log-counts of each token key, and *composition* features (leave-one-out mean of every other column inside each income-token group — what kind of rows the generator put at this income).
* Every label-derived feature is built **outside the row's own fold**: outer-fold validation/test rows use the full outer-training fold, outer-training rows are 5-fold cross-fitted inside it.
* A **ridge logistic GLM** (LBFGS, strong-Wolfe line search, `L2` penalty) on the standardized basis.

My additions are marked `# [added]`: a decimal floor chain and the per-value means of the original 10k rows (each ≤ +0.00001 OOF), `L2 = 30` instead of 10 (+0.00001), and — the one that matters for Part B — **inner-CV logits for the training rows** (`MARGINS`), so that the GLM can be a base margin for a GBDT without leaking the fold's labels.

In [ ]:
# ---- tokenizer (internet needed for the first download); falls back to a decimal hierarchy if unavailable
try:
    from transformers import AutoTokenizer
    TOK = AutoTokenizer.from_pretrained("gpt2")
    tokenize = lambda s: TOK(" " + s).input_ids
    print("GPT-2 tokenizer:", [TOK.decode([t]) for t in tokenize("92887")])
except Exception as e:                       # decimal fallback: [leading 1 digit, leading 2 digits, rest]
    print("tokenizer unavailable, decimal fallback:", e)
    tokenize = lambda s: [int(s[0]), int(s[:2]), int(s[2:] or 0)]

L2 = 30.0                                     # [added] was 10.0
LBFGS_ITERS = 200
OUT_TEMPERATURE, OUT_OFFSET = 1.25, -1.0
FLOAT_COLS = ["Annual_Income_USD", "Daily_Commute_km", "Environmental_Concern_Level"]
INT_COLS = ["Age", "Number_of_Cars_Owned", "Charging_Stations_Near_Home", "Charging_Stations_Near_Work"]
CAT_COLS = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]
COLS = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned", "Charging_Stations_Near_Home",
        "Charging_Stations_Near_Work", "Environmental_Concern_Level"] + CAT_COLS

ntr, nte = len(train), len(test)
X = pd.concat([train[COLS], test[COLS]], ignore_index=True)
N = len(X)
FOLD_ID = np.full(ntr, -1, np.int8)
for k, (_, va) in enumerate(SPLITS):
    FOLD_ID[va] = k

In [ ]:
t0 = time.time()
inc_str = X.Annual_Income_USD.astype(np.int64).astype(str).to_numpy()
u_inc, inv_inc = np.unique(inc_str, return_inverse=True)
tok_ids = [tokenize(s) for s in u_inc]

def fz(values):
    return pd.factorize(values, sort=False)[0].astype(np.int64)

def qbin(values, q):
    ranks = pd.Series(values).rank(method="first")
    return fz(pd.qcut(ranks, q, labels=False, duplicates="drop").to_numpy())

K = {}
K["L1"] = fz(np.array([t[0] for t in tok_ids], dtype=np.int64)[inv_inc])
K["L2"] = fz(np.array([t[0] * 60000 + (t[1] if len(t) > 1 else -1) for t in tok_ids], dtype=np.int64)[inv_inc])
K["IV"] = fz(inc_str)
K["LAST"] = fz(np.array([len(t) * 60000 + t[-1] for t in tok_ids], dtype=np.int64)[inv_inc])
cm = X.Daily_Commute_km.to_numpy(np.float64)
K["CV"] = fz(cm)
K["CI"] = fz(np.floor(cm))
for c in COLS:
    if c not in ("Annual_Income_USD", "Daily_Commute_km"):
        K[c] = fz(X[c].to_numpy())
K["AGEDEC"] = fz((X.Age // 10).to_numpy())
K["CMTBIN"] = qbin(cm, 10)
K["SHB"] = fz(np.minimum(X.Charging_Stations_Near_Home, 6).to_numpy())
K["SWB"] = fz(np.minimum(X.Charging_Stations_Near_Work, 6).to_numpy())
K["INCQ"] = qbin(X.Annual_Income_USD.to_numpy(), 5)
K["INC20"] = qbin(X.Annual_Income_USD.to_numpy(), 20)
K["INC_MOD100"] = fz((X.Annual_Income_USD.astype(np.int64) % 100).to_numpy())
K["INC_DIV1000"] = fz((X.Annual_Income_USD.astype(np.int64) // 1000).to_numpy())
K["CELL"] = fz(((K["Subsidy_Available"] * 5 + K["Environmental_Concern_Level"]) * 3 + K["Range_Anxiety_Level"]) * 2 + K["Home_Charging_Possible"])

def cross(a, b):
    return fz(K[a] * (int(K[b].max()) + 1) + K[b])

FLAT, FAMILY = {}, {}
for lv in ["L1", "L2", "LAST", "IV", "CV", "CI", "INC_MOD100", "INC_DIV1000", "Age"]:
    for a in (5, 50):
        FLAT[f"r_{lv}_a{a}"] = (K[lv], a); FAMILY[f"r_{lv}_a{a}"] = "flat_rate"
for lv in ["L1", "L2", "LAST"]:
    for c in ["City_Type", "Home_Charging_Possible", "Subsidy_Available", "Environmental_Concern_Level", "Range_Anxiety_Level",
              "Gender", "Current_Car_Type", "Number_of_Cars_Owned", "AGEDEC", "CMTBIN", "SHB", "SWB"]:
        FLAT[f"tokx_{lv}_{c}"] = (cross(lv, c), 20); FAMILY[f"tokx_{lv}_{c}"] = "token_x_context"
for c in ["City_Type", "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level", "Environmental_Concern_Level", "INCQ"]:
    FLAT[f"cmtx_CI_{c}"] = (cross("CI", c), 20); FAMILY[f"cmtx_CI_{c}"] = "commute_x_context"
for c in ["Home_Charging_Possible", "City_Type", "Range_Anxiety_Level"]:
    FLAT[f"cmtx_CV_{c}"] = (cross("CV", c), 20); FAMILY[f"cmtx_CV_{c}"] = "commute_x_context"
for name, key, alpha in [("cell", K["CELL"], 5), ("cell_x_L1", cross("CELL", "L1"), 20),
                         ("cell_x_CI", cross("CELL", "CI"), 20), ("cell_x_INCQ", cross("CELL", "INCQ"), 20)]:
    FLAT[name] = (key, alpha); FAMILY[name] = "generator_formula_cell"
CATS = ["Environmental_Concern_Level", "Subsidy_Available", "Range_Anxiety_Level", "Home_Charging_Possible", "City_Type",
        "Current_Car_Type", "Number_of_Cars_Owned", "Gender", "AGEDEC", "SHB", "SWB"]
for c in CATS:
    for suffix, other in [("INC20", "INC20"), ("CMT10", "CMTBIN")]:
        FLAT[f"xc_{c}_{suffix}"] = (cross(c, other), 20); FAMILY[f"xc_{c}_{suffix}"] = "category_x_continuous_bin"
for i, c1 in enumerate(CATS):
    for c2 in CATS[i + 1:]:
        FLAT[f"xp_{c1}_{c2}"] = (cross(c1, c2), 20); FAMILY[f"xp_{c1}_{c2}"] = "category_pair"
K["CELLI"] = cross("CELL", "INC20")
for name, key in [("x3_CELL_INC20", K["CELLI"]), ("x3_CELL_CMT10", cross("CELL", "CMTBIN"))]:
    FLAT[name] = (key, 20); FAMILY[name] = "cell_x_bin"
for c in ["City_Type", "Current_Car_Type", "Number_of_Cars_Owned", "AGEDEC"]:
    FLAT[f"x4_CELLI_{c}"] = (cross("CELLI", c), 20); FAMILY[f"x4_CELLI_{c}"] = "cell_x_bin"

CHAINS = {f"chain_S{S}": ([K["L1"], K["L2"], K["IV"]], S) for S in (5, 20, 80)}
CHAINS.update({f"chainC_S{S}": ([K["CI"], K["CV"]], S) for S in (5, 20)})
# [added] decimal floor chain income//1000 -> //100 -> exact, its flat rates, and floor x context cells
inc_i = X.Annual_Income_USD.astype(np.int64).to_numpy()
K["F1000"], K["F100"] = fz(inc_i // 1000), fz(inc_i // 100)
for lv in ["F1000", "F100"]:
    for a in (5, 50):
        FLAT[f"r_{lv}_a{a}"] = (K[lv], a); FAMILY[f"r_{lv}_a{a}"] = "flat_rate_floor"
for S in (5, 20, 80):
    CHAINS[f"chainF_S{S}"] = ([K["F1000"], K["F100"], K["IV"]], S)
for c in ["City_Type", "Subsidy_Available", "Environmental_Concern_Level", "Range_Anxiety_Level"]:
    FLAT[f"tokx_F1000_{c}"] = (cross("F1000", c), 20); FAMILY[f"tokx_F1000_{c}"] = "floor_x_context"

LABEL_NAMES = list(FLAT) + [f"{n}_lv{i}" for n, (ks, _) in CHAINS.items() for i in range(len(ks))]
for n, (ks, _) in CHAINS.items():
    for i in range(len(ks)):
        FAMILY[f"{n}_lv{i}"] = "token_tree_beta_chain" if n.startswith("chain_") else ("commute_beta_chain" if n.startswith("chainC") else "floor_beta_chain")

# ---- label-free features
LF, LFN = [], []
for nm in ["IV", "L2", "L1", "CV", "LAST"]:
    LF.append(np.log1p(np.bincount(K[nm])[K[nm]])); LFN.append(f"cnt_{nm}"); FAMILY[f"cnt_{nm}"] = "count_train_test"
inc_f = X.Annual_Income_USD.to_numpy(np.float64)
env = X.Environmental_Concern_Level.to_numpy(np.float64)
subs = (X.Subsidy_Available == "Yes").to_numpy()
anx_formula = X.Range_Anxiety_Level.map({"Low": 0, "Medium": 1, "High": 3}).to_numpy(np.float64)
LF.append(1.2 * inc_f / 1e5 + 0.6 * env + 2 * subs - anx_formula); LFN.append("formula"); FAMILY["formula"] = "generator_formula"
for c in COLS[:7]:
    v = X[c].to_numpy(np.float64)
    LF.append(v); LFN.append(f"raw_{c}"); FAMILY[f"raw_{c}"] = "raw"
    for q in (0.1, 0.3, 0.5, 0.7, 0.9):
        knot = np.quantile(v, q)
        LF.append(np.maximum(v - knot, 0.0)); LFN.append(f"hinge_{c}_{q}"); FAMILY[f"hinge_{c}_{q}"] = "raw_hinge"
OH = pd.get_dummies(X[CAT_COLS + ["Environmental_Concern_Level", "Number_of_Cars_Owned"]].astype(str), dtype=np.float32)
for c in OH.columns:
    LFN.append(f"oh_{c}"); FAMILY[f"oh_{c}"] = "one_hot"
LF = np.column_stack(LF + [OH.to_numpy()]).astype(np.float32)

MIXV = {"env": env, "sub": subs.astype(float),
        "anx": X.Range_Anxiety_Level.map({"Low": 0, "Medium": 1, "High": 2}).to_numpy(np.float64),
        "home": (X.Home_Charging_Possible == "Yes").to_numpy(float),
        "urban": (X.City_Type == "Urban").to_numpy(float), "rural": (X.City_Type == "Rural").to_numpy(float),
        "suv": (X.Current_Car_Type == "SUV").to_numpy(float), "truck": (X.Current_Car_Type == "Truck").to_numpy(float),
        "sedan": (X.Current_Car_Type == "Sedan").to_numpy(float), "male": (X.Gender == "Male").to_numpy(float),
        "cars": X.Number_of_Cars_Owned.to_numpy(float), "age": X.Age.to_numpy(float),
        "sth": X.Charging_Stations_Near_Home.to_numpy(float), "stw": X.Charging_Stations_Near_Work.to_numpy(float),
        "cmt": cm, "inc": inc_f / 1e4}
ivcv = fz(np.char.add(np.char.add(inc_str.astype(str), "_"), X.Daily_Commute_km.astype(str).to_numpy()))
mix_keys = {"IV": K["IV"], "L2": K["L2"], "L1": K["L1"], "CV": K["CV"], "IVCV": ivcv}
MIX, MIXN = [], []
for kn, key in mix_keys.items():
    counts = np.bincount(key).astype(np.float64); n = counts[key]
    MIX.append(np.log(np.maximum(n, 1.0))); MIXN.append(f"mix_{kn}_logn")
    for vn, v in MIXV.items():
        if (kn in ("IV", "L2", "L1") and vn == "inc") or (kn == "CV" and vn == "cmt") or (kn == "IVCV" and vn in ("inc", "cmt")):
            continue
        global_mean = float(v.mean()); sums = np.bincount(key, weights=v, minlength=len(counts))[key]
        MIX.append((sums - v + 5 * global_mean) / (n - 1 + 5) - global_mean); MIXN.append(f"mix_{kn}_{vn}")
for name in MIXN:
    FAMILY[name] = "composition_mix_transductive"
LF = np.concatenate([LF, np.column_stack(MIX).astype(np.float32)], axis=1); LFN += MIXN
# [added] per-value buyer rate in the original 10k rows (label-free with respect to the competition labels)
om = orig["_y"].mean()
for c in ["Annual_Income_USD", "Daily_Commute_km", "Age", "Environmental_Concern_Level"] + CAT_COLS:
    v = X[c].map(orig.groupby(c)["_y"].mean()).fillna(om).to_numpy(np.float32)
    LF = np.concatenate([LF, v[:, None]], axis=1); LFN.append(f"orig_{c}"); FAMILY[f"orig_{c}"] = "orig_mean"
v = X["Annual_Income_USD"].isin(set(orig["Annual_Income_USD"])).to_numpy(np.float32)
LF = np.concatenate([LF, v[:, None]], axis=1); LFN.append("orig_inc_seen"); FAMILY["orig_inc_seen"] = "orig_mean"
ALL_NAMES = LABEL_NAMES + LFN
P = len(ALL_NAMES)
print(f"GLM basis: {P} features | label-derived={len(LABEL_NAMES)} | label-free={len(LFN)}  [{time.time() - t0:.0f}s]")

In [ ]:
def _stats(key, fit):
    size = int(key.max()) + 1
    return np.bincount(key[fit], weights=y[fit], minlength=size), np.bincount(key[fit], minlength=size)

def _logit_rate(rate):
    rate = np.clip(rate, 1e-6, 1 - 1e-6)
    return np.log(rate / (1 - rate)).astype(np.float32)

def encode(fit, app, out, rows):
    # buyer rates of every FLAT key and every CHAIN level, fitted on `fit` rows, written for `app` rows
    prior = float(y[fit].mean()); j = 0
    for _, (key, alpha) in FLAT.items():
        s, c = _stats(key, fit); ka = key[app]
        out[rows, j] = _logit_rate((s[ka] + alpha * prior) / (c[ka] + alpha)); j += 1
    for _, (keys, strength) in CHAINS.items():
        post = np.full(len(app), prior, np.float64)
        for key in keys:
            s, c = _stats(key, fit); ka = key[app]
            post = (s[ka] + strength * post) / (c[ka] + strength)
            out[rows, j] = _logit_rate(post); j += 1

def build_fold(k):
    fit_rows, valid_rows = SPLITS[k]
    test_rows = np.arange(ntr, N)
    Mfit = np.empty((len(fit_rows), P), np.float32); Mvalid = np.empty((len(valid_rows), P), np.float32); Mtest = np.empty((nte, P), np.float32)
    n_label = len(LABEL_NAMES)
    encode(fit_rows, valid_rows, Mvalid, slice(None))
    encode(fit_rows, test_rows, Mtest, slice(None))
    position = np.full(ntr, -1, np.int64); position[fit_rows] = np.arange(len(fit_rows))
    inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=SEED)
    for inner_fit_rel, inner_app_rel in inner.split(np.zeros(len(fit_rows)), y[fit_rows]):
        encode(fit_rows[inner_fit_rel], fit_rows[inner_app_rel], Mfit, position[fit_rows[inner_app_rel]])
    Mfit[:, n_label:] = LF[fit_rows]; Mvalid[:, n_label:] = LF[valid_rows]; Mtest[:, n_label:] = LF[ntr:]
    return fit_rows, valid_rows, Mfit, Mvalid, Mtest

def fit_glm(Z, yy, l2=L2, iters=LBFGS_ITERS):
    w = torch.zeros(Z.shape[1], device=Z.device, dtype=torch.float64, requires_grad=True)
    b = torch.zeros(1, device=Z.device, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([w, b], lr=1.0, max_iter=iters, history_size=20, tolerance_grad=1e-9,
                            tolerance_change=1e-12, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad()
        loss = torch.nn.functional.binary_cross_entropy_with_logits(Z @ w + b, yy, reduction="sum") + 0.5 * l2 * (w * w).sum()
        loss.backward(); return loss
    opt.step(closure)
    return w.detach(), b.detach()

def to_probability(eta):
    return 1.0 / (1.0 + np.exp(-np.clip(OUT_TEMPERATURE * eta + OUT_OFFSET, -40, 40)))

In [ ]:
glm_oof = np.full(ntr, np.nan); glm_test_folds = []; coef_rows = []
MARGINS = {}                                  # [added] logits for Part B: fit rows inner-CV, valid/test rows from the full outer fit
for k in range(FOLDS):
    t0 = time.time()
    fit_rows, valid_rows, Mfit, Mvalid, Mtest = build_fold(k)
    mu = Mfit.mean(axis=0, dtype=np.float64); sd = Mfit.std(axis=0, dtype=np.float64) + 1e-6
    Zfit = torch.from_numpy((Mfit - mu) / sd).to(DEVICE, torch.float64)
    yy = torch.from_numpy(y[fit_rows].astype(np.float64)).to(DEVICE)
    w, b = fit_glm(Zfit, yy); del Zfit, yy
    Zvalid = torch.from_numpy((Mvalid - mu) / sd).to(DEVICE, torch.float64); Ztest = torch.from_numpy((Mtest - mu) / sd).to(DEVICE, torch.float64)
    eta_valid = (Zvalid @ w + b).cpu().numpy(); eta_test = (Ztest @ w + b).cpu().numpy(); del Zvalid, Ztest
    # [added] inner cross-fitted logits for the fit rows (never fitted on themselves)
    eta_fit = np.zeros(len(fit_rows))
    for ifit, iapp in StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=SEED + 1).split(np.zeros(len(fit_rows)), y[fit_rows]):
        mu_i = Mfit[ifit].mean(axis=0, dtype=np.float64); sd_i = Mfit[ifit].std(axis=0, dtype=np.float64) + 1e-6
        Zi = torch.from_numpy((Mfit[ifit] - mu_i) / sd_i).to(DEVICE, torch.float64); yi = torch.from_numpy(y[fit_rows][ifit].astype(np.float64)).to(DEVICE)
        wi, bi = fit_glm(Zi, yi); del Zi, yi
        Za = torch.from_numpy((Mfit[iapp] - mu_i) / sd_i).to(DEVICE, torch.float64); eta_fit[iapp] = (Za @ wi + bi).cpu().numpy(); del Za
    MARGINS[k] = (fit_rows, valid_rows, eta_fit, eta_valid, eta_test)
    glm_oof[valid_rows] = to_probability(eta_valid); glm_test_folds.append(to_probability(eta_test))
    coef_rows.extend({"feature": n, "family": FAMILY[n], "abs_coef": float(abs(c))} for n, c in zip(ALL_NAMES, w.cpu().numpy()))
    print(f"GLM fold {k + 1}/{FOLDS} | AUC={roc_auc_score(y[valid_rows], eta_valid):.5f} | inner-CV fit-row AUC={roc_auc_score(y[fit_rows], eta_fit):.5f} | {time.time() - t0:.0f}s", flush=True)
    del Mfit, Mvalid, Mtest, w, b; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
glm_test = np.mean(glm_test_folds, axis=0)
print(f"GLM pooled OOF AUC = {roc_auc_score(y, glm_oof):.5f}")
coef = pd.DataFrame(coef_rows)
print(coef.groupby("family").abs_coef.sum().sort_values(ascending=False).div(FOLDS).round(2).to_string())
print(coef.groupby("feature").abs_coef.mean().sort_values(ascending=False).head(12).round(3).to_string())
del LF, K, FLAT, CHAINS, X; gc.collect()

## Part B — Mine: LR-margin residual GBDT on 25 target-encoded keys

My feature set from the earlier notebook: the six categoricals and the exact value of every numeric column as string keys, ten digit-position keys of income and commute, three coarse keys (`income//100`, `income//1000`, `floor(commute)`), each key target-encoded inside the fold at `smooth ∈ {auto, 10, 100}` with sklearn's internal cross-fitting; train+test frequencies; per-value buyer rates from the original 10k rows; three generator-artifact flags.

The change: instead of blending the GLM with a GBDT trained from scratch, **the GBDT starts from the GLM's logit** — `init_score` in LightGBM, `base_margin` in XGBoost — and only learns what the linear model missed. Training rows get their *inner-CV* logit (Part A), validation and test rows get the outer fit's logit, so no fold sees its own labels through the margin. On these folds: plain LightGBM 0.94608 → residual LightGBM 0.94646; XGBoost 0.94610 → 0.94645; and the boosters stop after 400–1,300 trees instead of 1,500–2,500.

In [ ]:
def build_features(train, test, orig):
    n_train = len(train)
    df = pd.concat([train, test], ignore_index=True).drop(columns=["Number_of_Cars_Owned"])
    raw_cats = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]
    inc, com = df["Annual_Income_USD"], df["Daily_Commute_km"]
    df["inc_exact"] = inc.astype(int).astype(str); df["com_exact"] = com.round(1).astype(str)
    df["age_key"] = df["Age"].astype(str); df["env_key"] = df["Environmental_Concern_Level"].astype(int).astype(str)
    df["csh_key"] = df["Charging_Stations_Near_Home"].astype(str); df["csw_key"] = df["Charging_Stations_Near_Work"].astype(str)
    key_cols = raw_cats + ["inc_exact", "com_exact", "age_key", "env_key", "csh_key", "csw_key"]
    for c in ["Annual_Income_USD", "Daily_Commute_km"]:            # digit-position keys
        for k in range(-1, 6):
            d = (df[c] // (10 ** k) % 10).astype(int)
            if d.nunique() > 1:
                df[f"{c[:3]}_d{k}"] = d.astype(str); key_cols.append(f"{c[:3]}_d{k}")
    df["inc_100"] = (inc // 100).astype(int).astype(str); df["inc_1000"] = (inc // 1000).astype(int).astype(str)
    df["com_int"] = com.astype(int).astype(str)
    key_cols += ["inc_100", "inc_1000", "com_int"]
    fe = pd.DataFrame({f"{c}_fe": df[c].map(df[c].value_counts(normalize=True)).astype("float32") for c in key_cols})
    df = pd.concat([df, fe], axis=1)
    orig_mean = orig["_y"].mean()
    for c in ["Annual_Income_USD", "Daily_Commute_km", "Age", "Environmental_Concern_Level",
              "Charging_Stations_Near_Home", "Charging_Stations_Near_Work"] + raw_cats:
        df[f"{c}_orig_mean"] = df[c].map(orig.groupby(c)["_y"].mean()).fillna(orig_mean).astype("float32")
    df["is_30k_spike"] = (inc == 30000).astype("int8"); df["is_millionaire_cliff"] = (inc >= 170537).astype("int8")
    df["is_dead_zone"] = inc.between(38000, 42000).astype("int8")
    return df.iloc[:n_train].copy(), df.iloc[n_train:].copy(), key_cols

TE_SMOOTHS = ("auto", 10.0, 100.0)
def target_encode(X_tr, y_tr, X_others, key_cols):
    outs = [X_tr.copy()] + [X.copy() for X in X_others]
    for s in TE_SMOOTHS:
        te = TargetEncoder(cv=FOLDS, smooth=s, random_state=SEED)
        enc = [te.fit_transform(X_tr[key_cols], y_tr)] + [te.transform(X[key_cols]) for X in X_others]
        tag = "auto" if s == "auto" else str(int(s))
        for Xo, e in zip(outs, enc):
            Xo[[f"{c}_te{tag}" for c in key_cols]] = e.astype("float32")
    return [Xo.drop(columns=key_cols) for Xo in outs]

t0 = time.time()
tr_f, te_f, key_cols = build_features(train.drop(columns=[TARGET]), test, orig)
features = [c for c in te_f.columns if c != ID]
XB, XB_test = tr_f[features], te_f[features]
print(f"{len(features)} columns before TE, {len(key_cols)} keys x {len(TE_SMOOTHS)} smoothings  [{time.time() - t0:.0f}s]")

In [ ]:
LR = 0.02
def fit_lgbm(X_tr, y_tr, X_va, y_va, m_tr, m_va):
    clf = lgb.LGBMClassifier(n_estimators=20000, learning_rate=LR, max_depth=5, num_leaves=32, min_child_samples=10,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.3, reg_alpha=0.071, reg_lambda=2.0,
                             max_bin=1024, feature_pre_filter=False, random_state=SEED, n_jobs=-1, verbose=-1)
    clf.fit(X_tr, y_tr, init_score=m_tr, eval_set=[(X_va, y_va)], eval_init_score=[m_va], eval_metric="auc",
            callbacks=[lgb.early_stopping(500, verbose=False)])
    return (lambda Xq, m: 1.0 / (1.0 + np.exp(-(clf.predict(Xq, raw_score=True) + m)))), clf.best_iteration_

def fit_xgb(X_tr, y_tr, X_va, y_va, m_tr, m_va):
    from xgboost import XGBClassifier
    clf = XGBClassifier(n_estimators=20000, learning_rate=LR, max_depth=4, min_child_weight=5, subsample=0.9, colsample_bytree=0.6,
                        reg_lambda=5.0, max_bin=1024, tree_method="hist", device=XGB_DEVICE, eval_metric="auc",
                        early_stopping_rounds=500, random_state=SEED, n_jobs=-1, verbosity=0)
    clf.fit(X_tr, y_tr, base_margin=m_tr, eval_set=[(X_va, y_va)], base_margin_eval_set=[m_va], verbose=False)
    return (lambda Xq, m: 1.0 / (1.0 + np.exp(-clf.predict(Xq, output_margin=True, base_margin=m)))), clf.best_iteration

FITTERS = {"lgbm": fit_lgbm, "xgb": fit_xgb}
MODELS = ["lgbm", "xgb"]
gbdt_oof = {m: np.zeros(ntr) for m in MODELS}; gbdt_test = {m: np.zeros(nte) for m in MODELS}
for k in range(FOLDS):
    tr_idx, va_idx, m_tr, m_va, m_te = MARGINS[k]
    t0 = time.time()
    X_tr, X_va, X_te = target_encode(XB.iloc[tr_idx], y[tr_idx], [XB.iloc[va_idx], XB_test], key_cols)
    for m in MODELS:
        t1 = time.time()
        predict, iters = FITTERS[m](X_tr, y[tr_idx], X_va, y[va_idx], m_tr, m_va)
        gbdt_oof[m][va_idx] = predict(X_va, m_va); gbdt_test[m] += predict(X_te, m_te) / FOLDS
        print(f"  fold {k + 1} {m:4s} trees={iters:5d} AUC={roc_auc_score(y[va_idx], gbdt_oof[m][va_idx]):.5f} ({time.time() - t1:.0f}s; TE {t1 - t0:.0f}s)", flush=True)
for m in MODELS:
    print(f"residual {m:4s} OOF AUC = {roc_auc_score(y, gbdt_oof[m]):.5f}   (GLM alone {roc_auc_score(y, glm_oof):.5f})")
pd.DataFrame({ID: TRAIN_IDS, "glm": glm_oof, **{f"residual_{m}": gbdt_oof[m] for m in MODELS}}).to_csv("oof_mine.csv", index=False)
pd.DataFrame({ID: TEST_IDS, "glm": glm_test, **{f"residual_{m}": gbdt_test[m] for m in MODELS}}).to_csv("test_mine.csv", index=False)

## Part C — Referenced: public out-of-fold predictions as stack members

These are other people's models, used exactly as they published them (their notebook outputs / dataset are attached as inputs). What each brings to the stack, in their authors' words and my measurements (10-fold OOF unless noted; correlation with my residual LightGBM in rank space):

| Member | Author | OOF | ρ | Why it helps |
|---|---|---|---|---|
| Generator-aware ridge LR (10-fold) | heuljax | 0.94640 | 0.992 | the same basis as Part A, fitted on 90 % folds; a second seed for the linear view |
| XGB Sample: donor cross-fit basis (leave-self-out neighbour rates, hierarchical shrinkage, income-cell composition, gate-mixture posteriors), GAM `base_margin`, monotone rate columns | heuljax | 0.94631 | 0.995 | a richer, differently built rate basis |
| XGBoost + window encodings (centred income windows ±\$2…\$200, quantisation ladders, lift vs. original, `ECL×Subsidy×RA` trigram) | BlamerX | 0.94621 | 0.996 | sees income through windows rather than exact keys |
| View A (LightGBM, triple TE + smooth keys + digits, 3 seeds) and View D (no exact-income key, `//10 //50 //500 //5000` ladder) | megayak | 0.94628 / 0.94608 | 0.998 / 0.997 | D is weak alone but "sees the data differently" — it gets weight, A's XGBoost twin (View B) gets none |
| RealMLP, 3 seeds, retrained on the library's folds | yekenot (model), megayak (retrain) | 0.94618 | 0.996 | the only neural member |

Anything not attached is skipped with a warning.

In [ ]:
def aligned(path, col, ids):
    df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)
    s = df.set_index(ID)[col].reindex(ids)
    if s.isna().any():
        raise ValueError(f"{path}: {int(s.isna().sum())} ids missing")
    return s.to_numpy(np.float64)

PUBLIC = {  # name: (oof file, oof column, test file, test column, path hint)
    "heuljax_lr":  ("GENERATOR_AWARE_LOGREG_SAMPLE_OOF.parquet", "oof_pred", "GENERATOR_AWARE_LOGREG_SAMPLE_TEST.parquet", "test_pred", None),
    "heuljax_xgb": ("XGB_SAMPLE_OOF.parquet", "oof_pred", "XGB_SAMPLE_TEST.parquet", "test_pred", None),
    "blamerx_xgb": ("oof.csv", "pred", "submission.csv", TARGET, "window-encodings"),
    "megayak_A":   ("oof_six_views.csv", "A_lgbm_triple_te_digits_3seed", "test_six_views.csv", "A_lgbm_triple_te_digits_3seed", None),
    "megayak_D":   ("oof_six_views.csv", "D_no_exact_key_ladder_windows", "test_six_views.csv", "D_no_exact_key_ladder_windows", None),
    "realmlp_3seed": ("oof_realmlp_g.csv", "G_realmlp_3seed", "test_realmlp_g.csv", "G_realmlp_3seed", None),
}
OOF = {"residual_lgbm": gbdt_oof["lgbm"], "residual_xgb": gbdt_oof["xgb"], "glm_5fold": glm_oof}
TST = {"residual_lgbm": gbdt_test["lgbm"], "residual_xgb": gbdt_test["xgb"], "glm_5fold": glm_test}
for name, (fo, co, ft, ct, hint) in PUBLIC.items():
    try:
        OOF[name] = aligned(find_file(fo, hint), co, TRAIN_IDS); TST[name] = aligned(find_file(ft, hint), ct, TEST_IDS)
    except Exception as e:
        print(f"skipping {name}: {e}")
for name in OOF:
    print(f"{name:16s} OOF AUC {roc_auc_score(y, OOF[name]):.5f}   rho vs residual_lgbm {spearmanr(OOF[name], OOF['residual_lgbm'])[0]:.4f}")

## Part D — Mine: honest rank stacking, nested weights, boundary rules

* Every member is converted to percentile ranks (the public files come at different fold counts and calibrations; ranks make them commensurable and AUC only sees order).
* Weights are found by **greedy coordinate ascent on OOF AUC** (step 0.02) with a Mann–Whitney AUC that costs one `argsort` per evaluation, so a full climb over a dozen members takes a minute.
* To keep the weights honest they are fitted **inside a 5-fold nested loop** and the five weight vectors are averaged; the held-out score of each nested fold, minus the best single member, is the gain you can expect on unseen data. In my runs that gain was +0.00021 … +0.00026, positive in every fold, and public-board moves tracked it (0.94670 → 0.94675).
* Four boundary rules move the rows the generator recipe makes certain (100 % / 0 % buyers in training) to the extreme ranks. They are worth +0.000002 on OOF — a formality, but a safe one.

In [ ]:
def rk(v):
    return rankdata(v) / len(v)

def fast_auc(yb, s):
    order = np.argsort(s, kind="stable"); r = np.empty(len(s)); r[order] = np.arange(1, len(s) + 1)
    n1 = yb.sum(); n0 = len(yb) - n1
    return (r[yb == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def hill(R, yb, names, step=0.02, iters=300):
    w = {n: 0.0 for n in names}
    best = max(names, key=lambda n: fast_auc(yb, R[n])); w[best] = 1.0
    vec, cur = R[best].copy(), fast_auc(yb, R[best])
    for _ in range(iters):
        cand = [(fast_auc(yb, vec + step * R[n]), n) for n in names]
        a, n = max(cand)
        if a <= cur + 1e-7: break
        w[n] += step; vec = vec + step * R[n]; cur = a
    s = sum(w.values()); return {n: w[n] / s for n in names}, cur

names = list(OOF)
R = {n: rk(OOF[n]) for n in names}; T = {n: rk(TST[n]) for n in names}
w_full, auc_full = hill(R, y, names)
print(f"full-data hill-climb OOF {auc_full:.5f}:", {k: round(v, 3) for k, v in sorted(w_full.items(), key=lambda x: -x[1]) if v > 0})
ws, gains = [], []
for i, (a_idx, b_idx) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(ntr), y)):
    wi, _ = hill({n: R[n][a_idx] for n in names}, y[a_idx], names); ws.append(wi)
    held = fast_auc(y[b_idx], sum(wi[n] * R[n][b_idx] for n in names))
    best_single = max(fast_auc(y[b_idx], R[n][b_idx]) for n in names)
    gains.append(held - best_single); print(f"nested fold {i}: best single {best_single:.5f} -> stack {held:.5f} ({held - best_single:+.5f})")
W = {n: float(np.mean([wi[n] for wi in ws])) for n in names}
print(f"mean nested gain {np.mean(gains):+.5f}; nested-averaged weights:", {k: round(v, 3) for k, v in sorted(W.items(), key=lambda x: -x[1]) if v > 0.005})
stack_oof = sum(W[n] * R[n] for n in names); stack_test = sum(W[n] * T[n] for n in names)
print(f"stack OOF AUC (nested-averaged weights) = {fast_auc(y, stack_oof):.5f}")

In [ ]:
def boundary_rules(df, p):
    inc, km = df.Annual_Income_USD.to_numpy(), df.Daily_Commute_km.to_numpy()
    nosub = (df.Subsidy_Available == "No").to_numpy(); env1 = (df.Environmental_Concern_Level == 1).to_numpy()
    anx = df.Range_Anxiety_Level.isin(["Medium", "High"]).to_numpy()
    r = rk(p)
    r[inc >= 170537] += 2                                    # always buys in training (393/393)
    r[(inc >= 31004) & (inc <= 41970)] -= 2                  # never buys (0/1257)
    r[km >= 83] -= 2                                         # never buys (0/186)
    r[(inc == 30000) & nosub & (env1 | anx)] -= 2            # never buys (0/7157)
    return rankdata(r, method="ordinal") / len(r)

print(f"OOF before rules {fast_auc(y, stack_oof):.6f}  after {fast_auc(y, boundary_rules(train, stack_oof)):.6f}")
final = boundary_rules(test, stack_test)
sub = pd.DataFrame({ID: TEST_IDS, TARGET: final})
sub.to_csv("submission.csv", index=False)
pd.DataFrame({ID: TRAIN_IDS, "stack": stack_oof, **{n: OOF[n] for n in names}}).to_csv("oof_stack.csv", index=False)
print(sub.head(), "\nsaved submission.csv, oof_stack.csv, oof_mine.csv, test_mine.csv")

## What I learned

**1. The generator left a tokenizer-shaped fingerprint.** I compared income hierarchies by the OOF AUC of a three-level beta chain (`level1 → level2 → exact value`, strength 80) on the competition labels:

| Hierarchy of the income string | chain AUC |
|---|---|
| **GPT-2 BPE tokens** (heuljax) | **0.71758** |
| `income//1000 → income//100 → exact` | 0.71577 |
| cl100k / o200k tokens (3-digit groups) | 0.71498 |
| leading digit → leading two digits → exact | 0.71093 |

GPT-2's irregular chunking (` 9|28|87` but ` 3|0000` and ` 17|05|37`) groups incomes better than any decimal rule. That is why the first-token rate is the GLM's largest coefficient, and why a linear model with this basis beats every tree model here.

**2. Where the last 0.0003 came from, on fixed folds.**

| Step | OOF | Δ |
|---|---|---|
| My 25-key LightGBM + XGBoost + focal blend | 0.94614 | |
| heuljax GLM (5 folds) | 0.94632 | +0.00018 |
| GBDT boosted from the GLM logit (Part B) | 0.94646 | +0.00014 |
| + public OOF members, nested hill-climb (Part D) | 0.94664 | +0.00018 |
| + heuljax-basis XGBoost with the GLM margin, 3-seed GLM, 3-seed MLP | 0.94667 | +0.00003 |

Things that did **not** move the GLM once ported (10-fold, each within ±0.00001): commute tokens, gate × token crosses, `L2` 3 → 100, an MLP on the same basis (0.94613; it takes 5 % of the stack), fold seeds. The basis is saturated in the same way the tree feature sets were.

**3. What the public board pays for.** Stack OOF gains transferred one-to-one (0.94662 → 0.94670, 0.94664 → 0.94675) while a further +0.00003 OOF read as −0.00001 — the board's resolution is about 0.00002. Every 0.9465+ public notebook I read was a re-blend of one lineage of files, and the best of those (0.94656) was explicitly produced by probing the public split; nothing here uses a public submission file as an anchor, so the private board should see the same ordering as the OOF.

**4. Pseudo-labels (from my earlier notebook)** raise CV by +0.0002 … +0.0004 through validation-label leakage and move the board by exactly 0; labels from a model that never saw the fold give +0.0000. Not used here.

## Acknowledgments

This notebook exists because these people shared their work, and the credit for the score belongs mostly to them:

* **[heuljax](https://www.kaggle.com/heuljax)** — the generator-aware ridge logistic regression and the XGB sample. Part A is a port of that notebook; the idea that a *linear* model on a tokenizer-aware basis could lead this competition is entirely theirs, and it changed how I read the data. Thank you for publishing it with a complete, honest cross-fitting protocol.
* **[BlamerX](https://www.kaggle.com/blamerx)** — XGBoost with window encodings, whose OOF and test files are a stack member here.
* **[megayak](https://www.kaggle.com/megayak)** — the six-feature-views OOF library (and the RealMLP retrain) built specifically so that others could weight members honestly, plus the write-ups on what the public split does and does not tell you.
* **[yekenot](https://www.kaggle.com/yekenot)** — the RealMLP that megayak retrained.
* **[Chris Deotte](https://www.kaggle.com/cdeotte)** — the EDA that recovered the generator recipe and the boundary thresholds.
* **[taeyangg4](https://www.kaggle.com/taeyangg4)**, **[lucifer19](https://www.kaggle.com/lucifer19)**, **[Naji](https://www.kaggle.com/najiama)**, **[Georgy Mamarin](https://www.kaggle.com/georgymamarin)**, **[Den Pugovkin](https://www.kaggle.com/denpugovkin)** — the boundary-rule shifts, the OOF hub, and the analyses of what the board rewards that shaped the choices above.

If you fork this, please upvote their notebooks first. Thank you all — and thanks for reading.